In [1]:
from pathlib import Path

from IPython.display import display

from searchlight.lightcurves import load
from searchlight.review_exports import (
    DEFAULT_METADATA_CACHE_PATH,
    DEFAULT_RESULT_CACHE_META_PATH,
    DEFAULT_RESULT_CACHE_PATH,
    enrich_results_with_metadata,
    load_lensing_results,
)
from searchlight.review_widgets import make_zooniverse_review_item
from searchlight.zooniverse_results import (
    DEFAULT_SUBJECT_METADATA_CACHE_PATH,
    DEFAULT_TASK_KEY,
    add_candidate_ids,
    annotate_most_likely_answers,
    load_zooniverse_results,
    select_zooniverse_answer_results,
)


In [2]:
RESULTS_PATH = Path('data/results/simulens1.0_tess_spoc')
RESULT_CACHE_PATH = DEFAULT_RESULT_CACHE_PATH
RESULT_CACHE_META_PATH = DEFAULT_RESULT_CACHE_META_PATH
RESULT_METADATA_CACHE_PATH = DEFAULT_METADATA_CACHE_PATH

# CSV exported from Zooniverse/aggregation, containing integer subject_id and data.most_likely.
ZOONIVERSE_RESULTS_CSV = Path('data/zooniverse/check_peaks_results.csv')
SUBJECT_METADATA_CACHE_PATH = DEFAULT_SUBJECT_METADATA_CACHE_PATH

# Set this explicitly if the CSV does not contain a workflow_id column.
WORKFLOW_ID = None
TASK_KEY = DEFAULT_TASK_KEY
MOST_LIKELY_ANSWER = 'Three or more peaks'
TOP_N = 100


In [3]:
# Zooniverse CSV loading, workflow answer lookup, subject metadata mapping,
# and review-card rendering live in searchlight modules so notebooks and
# exporters share the same implementation.


In [4]:
# Review widget implementation: searchlight.review_widgets.make_zooniverse_review_item


In [5]:
results = load_lensing_results(
    results_path=RESULTS_PATH,
    cache_path=RESULT_CACHE_PATH,
    cache_meta_path=RESULT_CACHE_META_PATH,
)
results = add_candidate_ids(results)
print(f'Loaded {len(results)} lensing model rows from {RESULTS_PATH}')


Loading cached results from data/results/simulens1.0_tess_spoc/top_resnet_results_cache.pkl
Loaded 2160368 lensing model rows from data/results/simulens1.0_tess_spoc


In [6]:
zooniverse = load_zooniverse_results(ZOONIVERSE_RESULTS_CSV)
zooniverse, workflow_id, answer_labels = annotate_most_likely_answers(
    zooniverse,
    workflow_id=WORKFLOW_ID,
    task_key=TASK_KEY,
)
zooniverse['most_likely_answer'].value_counts(dropna=False)


Workflow 31708 T0 answers: {0: 'None, it just contains noise or stellar variability', 1: 'One peak', 2: 'Two peaks', 3: 'Three or more peaks'}


most_likely_answer
NaN                                                    2300
One peak                                               1751
None, it just contains noise or stellar variability    1210
Two peaks                                               182
Three or more peaks                                      53
Name: count, dtype: int64

In [7]:
selected = select_zooniverse_answer_results(
    results,
    zooniverse,
    MOST_LIKELY_ANSWER,
    subject_metadata_cache_path=SUBJECT_METADATA_CACHE_PATH,
)
review_results = selected.head(TOP_N).copy()
review_results[['subject_id', 'most_likely_answer', 'data.num_votes', 'data.agreement', 'candidate_id', 'path', 'lensing']].head()


Fetching Zooniverse subjects: 0subject [00:00, ?subject/s]

Selected 53 rows where most_likely_answer == 'Three or more peaks'
Mapped 53 selected subjects to paths


,subject_id,most_likely_answer,data.num_votes,data.agreement,candidate_id,path,lensing
0,119507226,Three or more peaks,1,1.0,d3ca8770-9e94-5781-9704-e9a05416199f,https://mast.stsci.edu/api/v0.1/Download/file/...,4.978697
1,119513747,Three or more peaks,1,1.0,b7411392-a6d9-53eb-8ae1-6301a28f720d,https://mast.stsci.edu/api/v0.1/Download/file/...,4.645565
2,119513830,Three or more peaks,1,1.0,28f25bb4-7d34-599f-ba99-e7178f91bdb7,https://mast.stsci.edu/api/v0.1/Download/file/...,4.642256
3,119516012,Three or more peaks,1,0.5,d0b8b7d1-f7c5-5947-b3e8-e1ad74da3547,https://mast.stsci.edu/api/v0.1/Download/file/...,4.558604
4,119522742,Three or more peaks,1,1.0,f5746f8e-7168-5b4c-89c4-1c24b6ba04a2,https://mast.stsci.edu/api/v0.1/Download/file/...,4.394792


In [8]:
review_export = enrich_results_with_metadata(
    review_results,
    cache_path=RESULT_METADATA_CACHE_PATH,
)
review_export[[
    'subject_id',
    'most_likely_answer',
    'data.num_votes',
    'data.agreement',
    'path',
    'lensing',
    'tic',
    'sector',
    'simbad_url',
]].head()


,subject_id,most_likely_answer,data.num_votes,data.agreement,path,lensing,tic,sector,simbad_url
0,119507226,Three or more peaks,1,1.0,https://mast.stsci.edu/api/v0.1/Download/file/...,4.978697,437060307,46,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
1,119513747,Three or more peaks,1,1.0,https://mast.stsci.edu/api/v0.1/Download/file/...,4.645565,284613078,33,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
2,119513830,Three or more peaks,1,1.0,https://mast.stsci.edu/api/v0.1/Download/file/...,4.642256,63664338,33,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
3,119516012,Three or more peaks,1,0.5,https://mast.stsci.edu/api/v0.1/Download/file/...,4.558604,287425173,33,https://simbad.cds.unistra.fr/simbad/sim-coo?C...
4,119522742,Three or more peaks,1,1.0,https://mast.stsci.edu/api/v0.1/Download/file/...,4.394792,15853326,70,https://simbad.cds.unistra.fr/simbad/sim-coo?C...


In [9]:
for row in review_results.to_dict('records'):
    lc = load(row['path'])
    display(make_zooniverse_review_item(row, lc))


4% (145/3707) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:4% (145/3707) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).


0% (10/3495) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (10/3495) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).


0% (10/3495) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (10/3495) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).


0% (10/3495) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (10/3495) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).


1% (142/10730) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:1% (142/10730) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).


...

4% (461/10358) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:4% (461/10358) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).


0% (9/11775) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (9/11775) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).


0% (11/3354) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:0% (11/3354) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).


1% (17/3278) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:1% (17/3278) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).


2% (248/11190) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
INFO:lightkurve.utils:2% (248/11190) of the cadences will be ignored due to the quality mask (quality_bitmask=17087).
